# ComfyUI Zero-Disk Launcher (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/j7708git/colab-workbench/blob/main/comfyui/comfyui.ipynb)

本 Notebook 會在 Colab VM 自動部署 ComfyUI、以 `hf_xet` 下載預設模型套件，並透過 **RAM Disk（`/dev/shm`）零磁碟輸出** 與 **Cloudflare Quick Tunnel** 提供安全 HTTPS 連線網址：

- **零磁碟落盤**：`--output-directory /dev/shm/comfy-out` 與 `--temp-directory /dev/shm/comfy-tmp` 讓生圖僅暫存於記憶體，不寫入 Colab 虛擬硬碟，也不掛載 Google Drive。
- **自動記憶體清理**：背景定期清除過期暫存圖檔，防止長期生圖塞滿 RAM。
- **內建公開模型套件**：
  - `qwen21_turbo_int8`：`Comfy-Org/Qwen-Image-2.1`（主模型 + `qwen3vl_8b_int8` + VAE）
  - `moss_sft_int8`：`Comfy-Org/MOSS-Image`（主模型 + `qwen3.5_9b_int8` + `flux2-vae`）＋ `Kijai/MOSS-image_experimental` 4 步蒸餾 LoRA
  - `both`（預設）：一次下載兩組完整套件（共約 33.5 GB）

In [ ]:
#@title 1. 同步 colab-workbench 並載入選填 HF_TOKEN
import importlib
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/j7708git/colab-workbench.git"
REPO_DIR = "/content/colab-workbench"
FALLBACK_BRANCH = "feat/comfyui-zero-disk-bootstrap"
BRANCH = "feat/comfyui-zero-disk-bootstrap"  #@param {type:"string"}
BRANCH = (BRANCH or FALLBACK_BRANCH).strip() or FALLBACK_BRANCH

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)

subprocess.run(
    ["git", "-C", REPO_DIR, "fetch", "origin", f"+refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}"],
    check=True,
)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)

init_path = os.path.join(REPO_DIR, "comfyui", "__init__.py")
if not os.path.isfile(init_path) and BRANCH != FALLBACK_BRANCH:
    print(f"[comfyui] 分支 '{BRANCH}' 尚未包含 comfyui/__init__.py，自動切換至 {FALLBACK_BRANCH} ...")
    BRANCH = FALLBACK_BRANCH
    subprocess.run(
        ["git", "-C", REPO_DIR, "fetch", "origin", f"+refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}"],
        check=True,
    )
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)

assert os.path.isfile(init_path), f"找不到 {init_path}，請確認分支 {BRANCH} 內容是否完整"

sys.path[:] = [REPO_DIR, *[p for p in sys.path if p != REPO_DIR]]
for _mod in [
    k
    for k in list(sys.modules)
    if k in {"comfyui", "shared"} or k.startswith(("comfyui.", "shared."))
]:
    del sys.modules[_mod]
importlib.invalidate_caches()

import comfyui
from shared import export_secret_to_env

assert getattr(comfyui, "__file__", None) is not None and hasattr(comfyui, "download_preset_models"), (
    f"comfyui 載入異常（__file__={getattr(comfyui, '__file__', None)}）"
)
head_sha = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], text=True).strip()
hf_token = export_secret_to_env("HF_TOKEN", required=False)
print("BRANCH:", BRANCH, f"({head_sha})")
print("comfyui.__file__:", comfyui.__file__)
print("HF_TOKEN:", "已載入" if hf_token else "未設定（公開模型可免 Token 下載）")

In [ ]:
#@title 2. 安裝／更新 ComfyUI 與零磁碟環境
!bash /content/colab-workbench/comfyui/install.sh

In [ ]:
#@title 3. 下載預設模型套件（Qwen-Image-2.1 Turbo int8 / MOSS-Image SFT int8 + 4-step LoRA）
import importlib
import sys

REPO_DIR = "/content/colab-workbench"
sys.path[:] = [REPO_DIR, *[p for p in sys.path if p != REPO_DIR]]
_cached_comfyui = sys.modules.get("comfyui")
if _cached_comfyui is not None and (
    getattr(_cached_comfyui, "__file__", None) is None
    or not hasattr(_cached_comfyui, "download_preset_models")
):
    for _mod in [k for k in list(sys.modules) if k == "comfyui" or k.startswith("comfyui.")]:
        del sys.modules[_mod]
importlib.invalidate_caches()

PRESET = "both"  #@param ["both", "qwen21_turbo_int8", "moss_sft_int8"]
EXTRA_MODELS_YAML = ""  #@param {type:"string"}

from comfyui import download_preset_models, load_models_from_yaml

extra = load_models_from_yaml(EXTRA_MODELS_YAML) if EXTRA_MODELS_YAML.strip() else None
downloaded = download_preset_models(preset=PRESET, extra_entries=extra)
print(f"模型就緒（共 {len(downloaded)} 個檔案）:")
for p in downloaded:
    print(" -", p)

In [ ]:
#@title 4. 啟動零磁碟 ComfyUI（/dev/shm）與 Cloudflare Tunnel
import importlib
import shlex
import sys

REPO_DIR = "/content/colab-workbench"
sys.path[:] = [REPO_DIR, *[p for p in sys.path if p != REPO_DIR]]
_cached_comfyui = sys.modules.get("comfyui")
if _cached_comfyui is not None and (
    getattr(_cached_comfyui, "__file__", None) is None
    or not hasattr(_cached_comfyui, "launch_comfyui_zero_disk")
):
    for _mod in [k for k in list(sys.modules) if k == "comfyui" or k.startswith("comfyui.")]:
        del sys.modules[_mod]
importlib.invalidate_caches()

from comfyui import launch_comfyui_zero_disk

PORT = 8188  #@param {type:"integer"}
MAX_AGE_SEC = 300  #@param {type:"integer"}
REAPER_INTERVAL_SEC = 60  #@param {type:"integer"}
EXTRA_ARGS = ""  #@param {type:"string"}

session = launch_comfyui_zero_disk(
    port=PORT,
    max_age_sec=MAX_AGE_SEC,
    reaper_interval_sec=REAPER_INTERVAL_SEC,
    extra_args=shlex.split(EXTRA_ARGS) if EXTRA_ARGS.strip() else None,
)

print("=" * 60)
print("ComfyUI 零磁碟服務已上線！")
print("Output RAM Disk:", session.output_dir)
print("Temp RAM Disk:  ", session.temp_dir)
print("Public URL:     ", session.public_url)
print("請將上方 Public URL 貼入瀏覽器或 dsh-simple-tavern 設定頁的 ComfyUI URL。")
print("若要手動關閉服務與隧道，請執行：session.stop()")
print("=" * 60)